# Stage 2 duplex fine-tune -- Colab A100 (discrete-unit architecture)

Runs on `otoSpeech-full-duplex-task-oriented-20h` (gated dataset -- you need an approved HF token).

**Architecture note:** this version represents speech as discrete units (frozen WavLM-large + k-means(1000), both pretrained) and trains Qwen as an ordinary classifier via cross-entropy, instead of regressing continuous mel-spectrograms. This matches how validated full-duplex speech-LM systems (dGSLM, Moshi) are actually built, and the k-means quantizer + vocoder are pretrained/frozen specifically so this stays feasible on a single Colab A100 session -- see `speechcore/discrete_tokenizer.py`'s docstring for the full reasoning and literature citations.

**Order matters, and two sections below are hard gates, not suggestions:**
- Section 4 (Stage 0 round-trip): per the project spec, do not run training until you've personally listened to the reconstructed audio and confirmed it's intelligible. This now checks the discrete tokenizer+vocoder, not Vocos.
- Section 5 (channel roles): `train/stage2_duplex.py` will refuse to start unless you've confirmed which physical audio channel is the user vs. the agent and written that into the config.

Set Runtime -> Change runtime type -> A100 GPU before running anything below.

## 0. Sanity: confirm the A100 is actually attached

In [ ]:
!nvidia-smi

## 1. Persistent storage (Google Drive)

Colab's local disk is wiped every time the runtime recycles (routinely happens on 12-24h sessions). Processed audio and checkpoints go on Drive so a disconnect doesn't lose hours of preprocessing or training. Only raw intermediate caches should ever live on the ephemeral local disk.

**Reusing the same `S2S-Engine-runs` folder as the earlier continuous-mel architecture run, deliberately -- not a collision risk:**
- `processed/` (the preprocessed otoSpeech audio -- `speaker_0.wav`/`speaker_1.wav`/`manifest.json`) is pure raw data, independent of architecture. Whether it later gets fed through continuous WavLM+Vocos or this discrete WavLM-large+k-means pipeline happens at training time, not preprocessing time -- reusing it skips re-downloading/re-decoding ~20h of audio for free, with zero risk.
- `checkpoints/` is where the two architectures could actually collide, so it's kept separated by name: the old run lives at `checkpoints/stage2/`, this discrete run writes to `checkpoints/stage2_discrete/` (set in Section 7). Neither is ever read or written by the other's code path.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DRIVE_DIR = '/content/drive/MyDrive/S2S-Engine-runs'
os.makedirs(PROJECT_DRIVE_DIR, exist_ok=True)
os.makedirs(f'{PROJECT_DRIVE_DIR}/processed', exist_ok=True)
os.makedirs(f'{PROJECT_DRIVE_DIR}/checkpoints', exist_ok=True)
print('Drive mounted. Artifacts will persist under:', PROJECT_DRIVE_DIR)

## 2. Get the code

Clones from GitHub. If you haven't pushed the latest `duplex_speechlm/` changes (dataprep manifest support, Stage 2 training script, duplex fusion, mask-aware variable-length training, channel-role gate, inference script, etc.) to `origin/main` yet, push first -- this clones whatever is on GitHub right now, not your local working copy.

In [ ]:
!rm -rf /content/S2S-Engine  # safe to drop a previous failed/partial clone before retrying
!git clone https://github.com/AmulyaJain2004/S2S-Engine.git /content/S2S-Engine
%cd /content/S2S-Engine/duplex_speechlm

In [ ]:
import os

# HARD CHECK: fail loudly right here if the clone/cd above didn't actually
# land in the repo, instead of a confusing IndexError/FileNotFoundError
# three cells later. If this assert fails: the repo is still private (git
# clone over plain https needs a token unless it's public), or the clone
# cell above errored and you need to scroll up and read that error.
assert os.getcwd() == '/content/S2S-Engine/duplex_speechlm', f'Wrong directory: {os.getcwd()}'
assert os.path.exists('train/stage2_duplex.py'), 'train/stage2_duplex.py missing -- clone did not actually succeed'
print('OK -- repo present at', os.getcwd())

In [ ]:
!pip install -q -r requirements.txt
!pip install -q matplotlib pandas
# Colab preinstalls a torchao version too old for the peft version requirements.txt
# pulls in -- peft probes for torchao whenever it's present and raises instead of
# skipping if the version is incompatible. We don't use torchao anywhere in this
# pipeline, so just remove it rather than pin a version to chase compatibility.
!pip uninstall -y -q torchao
# speechbrain (new dependency, for the discrete-unit tokenizer/vocoder) pulls in its
# own dependency set -- if you hit an import/version error here, it's most likely a
# dependency-pin conflict of the same KIND as the torchao one above (a package Colab
# preinstalls at an incompatible version). Read the actual error and uninstall/pin
# the specific conflicting package the same way, rather than assuming the code is wrong.
import speechbrain
print('speechbrain', speechbrain.__version__, 'imported OK')

## 3. Hugging Face auth

Needs a token with approved access to `otoearth/otoSpeech-full-duplex-task-oriented-20h`. This will prompt for the token interactively -- paste it, don't hardcode it in a cell.

In [ ]:
from huggingface_hub import login
login()

## 4. STAGE 0 GATE -- preprocess one session, then listen

Pulls exactly one real session (so this is fast) and runs the discrete tokenizer+vocoder round trip (`scripts/p0_discrete_roundtrip.py`: WavLM-large -> k-means(1000) -> UnitHiFiGAN, all pretrained/frozen) on a 5s slice of it. **Listen to both audio players that appear below.** This vocoder was pretrained on LibriTTS/general speech, not on otoSpeech specifically -- if it sounds muffled or clearly unlike natural speech on real data from this dataset, that's a domain-mismatch risk worth knowing about now, before spending GPU time training on top of it.

In [ ]:
!python dataprep/preprocess_otospeech.py --output_dir /content/one_session_check --max_sessions 1

In [ ]:
import glob, soundfile as sf

session_dir = glob.glob('/content/one_session_check/session_*')[0]
wav_path = f'{session_dir}/speaker_0.wav'
data, sr = sf.read(wav_path)

# Start partway into the session, not at sample 0 -- real conversational
# recordings routinely have a few seconds of dead air / line noise / room
# tone before anyone speaks, and grabbing that as the P0 "original" clip
# looks exactly like a broken pipeline even when nothing is wrong. 60s in
# (or 30% of the session if it's shorter than ~200s) is far more likely to
# land on actual speech.
duration_s = len(data) / sr
offset_s = min(60.0, duration_s * 0.3)
start = int(offset_s * sr)
clip = data[start : start + int(5 * sr)]

clip_path = '/content/p0_input_clip.wav'
sf.write(clip_path, clip, sr)
print(f'Using clip: {clip_path} (from {offset_s:.1f}s into a {duration_s:.1f}s session)')

In [ ]:
!python scripts/p0_discrete_roundtrip.py --wav /content/p0_input_clip.wav

In [ ]:
from IPython.display import Audio, display
print('ORIGINAL:')
display(Audio('outputs/p0_discrete_original.wav'))
print('RECONSTRUCTED (WavLM-large -> k-means(1000) -> UnitHiFiGAN, all pretrained/frozen -- '
      'this is the quality CEILING Stage 2 generation can possibly reach, since this tokenizer is lossy):')
display(Audio('outputs/p0_discrete_reconstructed.wav'))

### STOP HERE until you've listened.

Only run the cells below once the reconstruction above sounds like intelligible speech (it will not be identical to the original -- this tokenizer+vocoder pair is lossy by design -- but it must not be noise or obviously muffled/unlike natural speech).

**If the reconstruction changes the actual CONTENT (a different sentence, not just worse audio quality) -- run Section 4b below before going any further.** That's a more serious finding than "sounds rough": it means the pretrained tokenizer+vocoder isn't preserving enough of this dataset's speech to train on at all, and no amount of Stage 2 training can fix that (Stage 2 only ever sees this tokenizer's own representation, never the raw audio).

## 4b. If content changed: layer sweep (only run if Section 4 failed the way described above)

`scripts/p0_layer_sweep.py` tests every WavLM-large layer speechbrain's pretrained k-means/vocoder set supports (1, 3, 7, 12, 18, 23) on the SAME clip, reusing one loaded WavLM-large encoder across all of them (not six separate 1.2GB downloads). It also saves the exact 16kHz-resampled audio speechbrain actually sees, so you can rule out our own resampling as the cause first.

It also prints each layer's **unique unit count** -- if a layer collapses to very few distinct units across the whole clip, that's mode collapse (most frames mapping to a handful of clusters), which would explain generic/unrelated output regardless of what was actually said.

In [ ]:
!python scripts/p0_layer_sweep.py --wav /content/p0_input_clip.wav --output_dir /content/layer_sweep

In [ ]:
import glob
from IPython.display import Audio, display

print('ORIGINAL:')
display(Audio('/content/layer_sweep/sweep_original.wav'))
print('RESAMPLED 16kHz INPUT (what speechbrain actually sees -- if THIS already sounds wrong,')
print('the bug is in our resampling, not the pretrained tokenizer):')
display(Audio('/content/layer_sweep/sweep_resampled_16k_input.wav'))

for path in sorted(glob.glob('/content/layer_sweep/sweep_layer*_reconstructed.wav')):
    layer = path.split('layer')[1].split('_')[0]
    print(f'LAYER {layer}:')
    display(Audio(path))

## 5. GATE -- confirm speaker-channel roles

`dataprep/print_channel_roles.py` prints the real `speakers_by_channel` metadata from this session (plus `task_info` and a labelled best-effort guess) so you can read it yourself.

**Heads up:** otoSpeech-task-oriented-20h may turn out to be peer-to-peer conversation data (both speakers generically labeled `"task_role_name": "speaker"`, no inherent "agent" vs "user" role) rather than a customer/agent transcript -- the script will print an explicit note if that's the case. If so, `user_channel`/`agent_channel` isn't "which one is really the user" (there may be no such thing); it's just a fixed, consistent choice of which channel the model learns to produce vs. treats as incoming audio. Any consistent choice is valid for Stage 2's actual goal (duplex turn-taking/backchannel timing) -- it just has to be the SAME channel index for every session, which the script's consistency check verifies. `train/stage2_duplex.py` still hard-refuses to start training until `audio.channel_roles_verified: true` is set in the config written in Section 7 -- this is not optional to skip, even when the choice is arbitrary, because it forces you to make the choice deliberately rather than by accident.

In [ ]:
!python dataprep/print_channel_roles.py --manifest /content/one_session_check/manifest.json --n_sessions 1

Read the printed JSON above yourself (the 'guess' is only a heuristic). Set the two variables below to match what you actually confirmed, then move on to Section 6 -- Section 7 uses these variables when it writes the training config and will NOT silently default to an unverified guess.

In [ ]:
# EDIT THESE after reading Section 5's output -- there is no safe default.
CONFIRMED_USER_CHANNEL = 0   # the channel you confirmed is the user/customer side
CONFIRMED_AGENT_CHANNEL = 1  # the channel you confirmed is the agent side
I_HAVE_VERIFIED_CHANNEL_ROLES = False  # set True only after you've actually read Section 5's output

## 6. Preprocess the full (or a capped subset of the) dataset onto Drive

This streams every session and writes `processed/session_*/` + `processed/manifest.json`. It's resumable: re-running with the same `--output_dir` skips sessions already in the manifest, so a Colab disconnect mid-preprocessing doesn't force a restart from zero.

Start with `--max_sessions 20` or so to get a first Stage 2 run going quickly; drop the flag (or raise it) later to use more of the 20h.

If your dataset has many sessions and/or speaker roles might not be assigned to the same channel every session, also widen the `--n_sessions` check in Section 5 against this full manifest once it exists (re-run that cell pointed at `$PROCESSED_DIR/manifest.json`) -- `print_channel_roles.py` will warn you explicitly if the channel-to-role mapping is inconsistent across sessions, in which case a single global `user_channel`/`agent_channel` is the wrong model and this needs revisiting before training.

In [ ]:
PROCESSED_DIR = f'{PROJECT_DRIVE_DIR}/processed'
!python dataprep/preprocess_otospeech.py --output_dir "$PROCESSED_DIR" --max_sessions 20

Re-run the cell above (bump `--max_sessions`, or drop it entirely) whenever you want more data processed -- it only does new sessions.

## 7. Write a Colab-specific config pointing at Drive paths

**This is a new architecture (discrete units, not continuous mel regression) -- any earlier checkpoint is NOT compatible.** This writes `checkpoint_dir` to a NEW path (`checkpoints/stage2_discrete`) specifically so training starts fresh. Earlier runs are still sitting untouched at their old paths on Drive if you want to keep them for comparison.

In [ ]:
import yaml

assert I_HAVE_VERIFIED_CHANNEL_ROLES, (
    'Section 5 was not confirmed -- read its output and flip I_HAVE_VERIFIED_CHANNEL_ROLES to True '
    'before writing a config. train/stage2_duplex.py will refuse to train otherwise anyway.'
)

with open('configs/stage2.yaml') as f:
    cfg = yaml.safe_load(f)

cfg['paths']['manifest_path'] = f'{PROCESSED_DIR}/manifest.json'
cfg['paths']['checkpoint_dir'] = f'{PROJECT_DRIVE_DIR}/checkpoints/stage2_discrete'  # NEW path -- see markdown above
cfg['audio']['user_channel'] = CONFIRMED_USER_CHANNEL
cfg['audio']['agent_channel'] = CONFIRMED_AGENT_CHANNEL
cfg['audio']['channel_roles_verified'] = True
# cfg['train']['batch_size'] = 4     # lower first if you hit OOM -- WavLM-large now runs on both streams

with open('configs/stage2_colab.yaml', 'w') as f:
    yaml.safe_dump(cfg, f)

print(yaml.safe_dump(cfg))

## 8. Train

Checkpoints + `train_log.csv` write to Drive every `save_every` steps, so if Colab disconnects, just re-run this cell -- it resumes from the last saved checkpoint automatically (see `try_resume` in `train/stage2_duplex.py`). Training is now classification (cross-entropy over 1000 discrete units), not regression -- `train_log.csv` logs unit-prediction accuracy alongside loss, which is a much more directly interpretable signal than mel L1 ever was (e.g. "40% accuracy" is meaningful on its own; "L1 loss 0.5" never was).

Tip: for a quick pipeline sanity check before committing to a long run, temporarily lower `max_steps` in `configs/stage2.yaml` (e.g. to 200) before writing Section 7's config, then run Section 10 below against that tiny checkpoint just to confirm generation runs end to end -- it won't sound good yet, but it proves nothing is structurally broken before you spend hours training for real.

In [ ]:
!python train/stage2_duplex.py --config configs/stage2_colab.yaml

## 9. Plot training loss (run anytime, even while training is still going in another cell)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

log = pd.read_csv(f'{PROJECT_DRIVE_DIR}/checkpoints/stage2_discrete/train_log.csv')
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(log['step'], log['loss']); ax1.set_xlabel('step'); ax1.set_ylabel('cross-entropy loss'); ax1.set_title('Loss')
ax2.plot(log['step'], log['accuracy']); ax2.set_xlabel('step'); ax2.set_ylabel('unit accuracy'); ax2.set_title('Unit-prediction accuracy')
plt.tight_layout(); plt.show()

## 10. Generate and listen (the real test)

High unit-accuracy doesn't by itself mean the output is intelligible or well-timed -- this is the only way to actually check. `eval/generate_stage2.py` loads the trained checkpoint, encodes the user audio into discrete units, and runs generation window-by-window with a KV-cached Qwen; predicted units are accumulated and decoded to waveform in ONE call at the end (the pretrained vocoder isn't chunk-limited the way the old continuous one was, so no crossfade-stitching is needed anymore). Feeds in a real user-channel clip it was never trained on and listens to what comes out.

**History, for context:** earlier continuous-mel versions of this architecture had a real training-design bug (duplex fusion let the model shortcut training by echoing ground-truth audio) plus a documented over-smoothing problem inherent to L1 mel regression. The discrete-unit pivot addresses both: there's no continuous target to average over, and classification doesn't have the over-smoothing failure mode regression does.

In [ ]:
!python dataprep/preprocess_otospeech.py --output_dir /content/holdout_check --max_sessions 1

The cell above streams the NEXT unseen session (it skips any session already in `/content/one_session_check` or your processed manifest by id, since `preprocess_all_sessions` keys on session_id across the whole stream -- if it happens to land on a session already used for training, just point `--user_wav` below at a different `processed/session_*/speaker_<CONFIRMED_USER_CHANNEL>.wav` from your full manifest instead).

In [ ]:
import glob

holdout_dir = glob.glob('/content/holdout_check/session_*')[0]
holdout_user_wav = f'{holdout_dir}/speaker_{CONFIRMED_USER_CHANNEL}.wav'
print('Using held-out user clip:', holdout_user_wav)

!python eval/generate_stage2.py \
    --config configs/stage2_colab.yaml \
    --user_wav "$holdout_user_wav" \
    --window_s 0.4 \
    --max_duration_s 30 \
    --out_dir /content/generation_check

In [ ]:
from IPython.display import Audio, display

print('INPUT (what the model heard):')
display(Audio('/content/generation_check/input_user.wav'))
print('GENERATED (windowed, KV-cached, same mechanism as training):')
display(Audio('/content/generation_check/generated_agent.wav'))

## 11. Sanity check: does windowed/KV-cached generation match a one-shot forward pass?

Training processes a whole chunk through Qwen in one non-cached forward call; `generate_stage2.py` processes the same kind of audio window-by-window with a KV cache. Those should be mathematically equivalent -- `eval/one_shot_check.py` verifies that cheaply on a held-out chunk, instead of just assuming it. If this one-shot output sounds fine but Section 10's windowed output doesn't, the bug is in the windowing/KV-cache logic specifically, not in training or model capacity.

In [ ]:
!python eval/one_shot_check.py \
    --config configs/stage2_colab.yaml \
    --session_dir "$holdout_dir" \
    --offset_s 60 \
    --duration_s 8.0 \
    --out_dir /content/one_shot_check

In [ ]:
from IPython.display import Audio, display

print('GROUND TRUTH, raw (real agent audio for this chunk):')
display(Audio('/content/one_shot_check/ground_truth_agent.wav'))
print('GROUND TRUTH, re-encoded through the SAME frozen tokenizer (the quality CEILING the prediction can reach):')
display(Audio('/content/one_shot_check/ground_truth_reencoded.wav'))
print('ONE-SHOT PREDICTION (same checkpoint, non-windowed, no KV cache -- compare to Section 10, and to the ceiling above):')
display(Audio('/content/one_shot_check/one_shot_predicted_agent.wav'))